# Maximum Eccentricity of Lateral Ventricles (MaxEccLV) — MRI Adaptation

Fits ellipses to the lateral ventricle contours on a coronal slice at the PC level and reports the maximum eccentricity.

**MRI adaptation**: No HU windowing. Same coronal segmentation pipeline as CallosalAngle_MRI, with ellipse fitting instead of angle measurement.

**Reference**: Samadani et al., JNS 2024

## Library imports

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import cv2 as cv
import SimpleITK as sitk
from skimage import measure, filters
import matplotlib.pyplot as plt

from mri_utils import (rotate_image, rotate_image_coronal, getLargestCC, extract_coronal_view,
                       moore_contour, normalize_to_uint8,
                       OASIS_AC_VOXEL, OASIS_PC_VOXEL, parse_acpc_coords, load_acpc_from_txt)

## Configuration

Set `DATASET` to match your data source. The notebook adapts file discovery and AC/PC loading accordingly.

Supported datasets:
- **`"oasis1"`** — OASIS-1 cross-sectional (skull-stripped, atlas-registered `.img` files). AC/PC loaded from known OASIS atlas voxel coordinates.
- **`"abcd"`** — ABCD Study T1w scans (ART-corrected `.nii`/`.nii.gz` files). AC/PC loaded from per-subject `.txt` files produced by Charlie's ART script.
- **`"custom"`** — Any NIfTI dataset; set `SCAN_GLOB` and provide AC/PC loading logic.

In [ ]:
# ── Choose your dataset ──────────────────────────────────────────────
DATASET = "oasis1"  # "oasis1", "abcd", or "custom"

# ── Dataset-specific settings ────────────────────────────────────────
if DATASET == "oasis1":
    DATA_DIR = "data/oasis1/disc1"
    DEMOGRAPHICS_PATH = "data/oasis1/oasis_cross-sectional-5708aa0a98d82080.xlsx"
    # OASIS skull-stripped, gain-field corrected, atlas-registered volumes
    SCAN_GLOB = "OAS1_*/PROCESSED/MPRAGE/T88_111/*_masked_gfc.img"
    SUBJECT_ID_FROM_PATH = lambda p: p.split(os.sep)[-5]  # e.g. OAS1_0001_MR1
    # OASIS is atlas-registered: AC and PC have known fixed voxel coordinates
    # OASIS_PC_VOXEL = (85, 87, 73) — defined in mri_utils
    ACPC_MODE = "oasis_atlas"  # use OASIS_PC_VOXEL directly

elif DATASET == "abcd":
    DATA_DIR = "data/abcd"
    DEMOGRAPHICS_PATH = None  # set path to ABCD demographics CSV/TSV if available
    # ART-corrected T1w scans
    SCAN_GLOB = "**/*_T1w_acpc.nii*"
    SUBJECT_ID_FROM_PATH = lambda p: os.path.basename(p).split("_ses")[0]  # e.g. sub-NDARINVXXXXXXXX
    # Per-subject AC/PC .txt file from Charlie's ART batch script
    ACPC_TXT_GLOB = "**/*_acpc_coords.txt"
    ACPC_MODE = "txt_file"

elif DATASET == "custom":
    DATA_DIR = "data/my_dataset"
    DEMOGRAPHICS_PATH = None
    SCAN_GLOB = "**/*.nii.gz"  # adjust to your file naming
    SUBJECT_ID_FROM_PATH = lambda p: os.path.basename(p).replace(".nii.gz", "").replace(".nii", "")
    ACPC_MODE = "txt_file"  # or "oasis_atlas" if atlas-registered

else:
    raise ValueError(f"Unknown DATASET: {DATASET!r}. Use 'oasis1', 'abcd', or 'custom'.")

OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Dataset:      {DATASET}")
print(f"Data dir:     {DATA_DIR}")
print(f"Scan pattern: {SCAN_GLOB}")
print(f"AC/PC mode:   {ACPC_MODE}")

## Processing parameters

Empirically determined parameters from the original CT pipeline. These control the coronal-plane segmentation and the downstream ventricle smoothing/masking steps.

`coronal_plane_processing_params` — controls the per-slice segmentation: Gaussian blur kernel, morphological close kernel and iterations, adaptive threshold neighbourhood and C constant, and small-blob suppression threshold.

`ventricles_processing_smoothing_params` — controls the post-segmentation ROI masking and dynamic median smoothing: the middle-ROI margins (pixels to expand from the brain centroid), percentile-based upper crop threshold, smoothing kernel size and decrement, and the minimum ventricle-to-brain ratio before smoothing is rolled back.

In [ ]:
coronal_plane_processing_params = {
    "coronal_initial_gauss_blur": 3,
    "coronal_close_kernel_size": 4,
    "coronal_close_num_iterations": 2,
    "coronal_adaptive_thresh_size": 75,
    "coronal_adaptive_thresh_mean_thresh": 7.5,
    "suppress_label_thresh": 0.05,
}

ventricles_processing_smoothing_params = {
    "middle_roi_upper_dec": 5,
    "middle_roi_lower_inc": 7,
    "middle_roi_left_dec": 20,
    "middle_roi_right_dec": 20,
    "scale_upper_thresh": 0.5,
    "smoothing_factor": 7,
    "dec_thresh": 0,
    "ventricle_to_brain_ratio_thresh": 0.015,
}

## Coronal segmentation: `segment_ventricles()`

Extracts the coronal plane at the PC level and segments the lateral ventricles.

**MRI adaptation vs CT original:**
- CT (`MaxEccLV.ipynb`): reads a separate coronal NIfTI file, calls `window_stack_sitk()` to clamp to brain HU range (0–80 HU), then extracts the coronal slice at the PC index.
- MRI (this notebook): receives a pre-rotated 3D numpy array `img_arr` directly. Extracts the coronal plane as `img_arr[:, pl, :]` using the PC index derived from the atlas voxel coordinate or the physical-to-index transform. **No windowing** — Otsu thresholding operates directly on raw MRI intensities.

Pipeline after extraction:
1. Gaussian blur (de-noising)
2. Otsu threshold → binary brain mask
3. Morphological close → fill small gaps in brain boundary
4. Floodfill from (0,0) → full brain mask
5. Adaptive threshold → fine-grained brain tissue mask
6. Suppress small spurious blobs
7. Morphological open with asymmetric kernel → break choroid plexus occlusions
8. Ventricles = full brain mask − brain tissue (dark regions inside brain)

In [ ]:
def segment_ventricles(img_arr, pc_index, params, visualize=False):
    """Segments lateral ventricles from the coronal plane at the PC level.

    MRI adaptation of the CT MaxEccLV segment_ventricles function.
    Takes a pre-rotated 3D numpy array and a PC voxel index instead of
    reading a coronal NIfTI file and calling window_stack_sitk().

    Parameters
    ----------
    img_arr : np.ndarray
        3D T1w MRI volume (already rotated to RAS), shape (Z, Y, X).
        Coronal slices are indexed along the Y axis (axis 1).
    pc_index : int or float
        Voxel index along the Y axis (coronal direction) of the
        Posterior Commissure.
    params : dict
        Processing parameters dict (coronal_plane_processing_params).
    visualize : bool, optional
        If True, displays intermediate processing steps (default False).

    Returns
    -------
    ventricles : np.ndarray
        Binary mask of the segmented lateral ventricles (2D, coronal plane).
    full_brain_mask : np.ndarray
        Binary mask of the full brain (2D, coronal plane).
    cor_pc_index : int
        Rounded integer Y index used to extract the coronal slice.
    coronal_plane : np.ndarray
        Raw coronal slice at the PC level (float32, no windowing applied).
    """
    # Unpack processing parameters
    coronal_initial_gauss_blur = params["coronal_initial_gauss_blur"]
    coronal_close_kernel_size = params["coronal_close_kernel_size"]
    coronal_close_num_iterations = params["coronal_close_num_iterations"]
    coronal_adaptive_thresh_size = params["coronal_adaptive_thresh_size"]
    coronal_adaptive_thresh_mean_thresh = params["coronal_adaptive_thresh_mean_thresh"]
    suppress_label_thresh = params["suppress_label_thresh"]

    # ── MRI adaptation: extract coronal slice directly from 3D array ──
    # CT pipeline reads a separate coronal volume file and calls window_stack_sitk().
    # For T1w MRI we index the coronal plane from the already-rotated 3D array.
    # img_arr shape is (Z, Y, X); axis 1 is the anterior-posterior (coronal) axis.
    cor_pc_index = int(np.round(pc_index))
    coronal_plane = img_arr[:, cor_pc_index, :].astype(np.float32)
    # ──────────────────────────────────────────────────────────────────

    # Normalize to uint8 range for OpenCV operations
    plane_min, plane_max = coronal_plane.min(), coronal_plane.max()
    if plane_max > plane_min:
        coronal_plane_u8 = np.uint8(
            255.0 * (coronal_plane - plane_min) / (plane_max - plane_min)
        )
    else:
        coronal_plane_u8 = np.zeros_like(coronal_plane, dtype=np.uint8)

    # Step 1: Gaussian blur (de-noising)
    coronal_gauss_blurred = cv.GaussianBlur(
        coronal_plane_u8,
        (coronal_initial_gauss_blur, coronal_initial_gauss_blur),
        cv.BORDER_DEFAULT,
    )

    # Step 2: Otsu threshold
    otsu_thresh = filters.threshold_otsu(coronal_gauss_blurred)
    coronal_bin = coronal_gauss_blurred > otsu_thresh

    # Step 3: Morphological close
    close_kernel = cv.getStructuringElement(
        cv.MORPH_ELLIPSE, (coronal_close_kernel_size, coronal_close_kernel_size)
    )
    closed_coronal_brain_mask = cv.morphologyEx(
        np.uint8(coronal_bin),
        cv.MORPH_CLOSE,
        close_kernel,
        iterations=coronal_close_num_iterations,
    )

    # Step 4: Floodfill from (0,0) → full brain mask
    im_th = coronal_bin.copy()
    h, w = im_th.shape[:2]
    mask = np.zeros((h + 2, w + 2), np.uint8)
    im_floodfill = im_th.copy()
    res = cv.floodFill(np.uint8(im_floodfill), mask, (0, 0), 255)
    full_brain_mask = (1 - res[2]).copy()
    full_brain_mask = full_brain_mask[1:h + 1, 1:w + 1]

    # Brain centroid (for middle ROI in process_ventricles)
    M = cv.moments(np.uint8(full_brain_mask))
    if M["m00"] != 0:
        bcX = int(M["m10"] / M["m00"])
        bcY = int(M["m01"] / M["m00"])
    else:
        bcX, bcY = full_brain_mask.shape[1] // 2, full_brain_mask.shape[0] // 2

    # Step 5: Adaptive threshold
    coronal_adapt_thresh = cv.adaptiveThreshold(
        coronal_gauss_blurred, 1,
        cv.ADAPTIVE_THRESH_MEAN_C,
        cv.THRESH_BINARY,
        coronal_adaptive_thresh_size,
        coronal_adaptive_thresh_mean_thresh,
    )
    coronal_adapt_thresh_bg_suppressed = np.where(
        (1 - full_brain_mask) > 0, 0, coronal_adapt_thresh
    )
    coronal_brain_mask = np.where(coronal_adapt_thresh_bg_suppressed > 0, 1, 0)

    # Step 6: Suppress small spurious blobs
    blobs_before_processing = measure.label(
        full_brain_mask - coronal_brain_mask, background=0
    )
    try:
        coronal_brain_mask_small_holes_closed = coronal_brain_mask.copy()
        coronal_bin_closed = coronal_bin.copy()
        blob_sums = np.unique(blobs_before_processing, return_counts=True)[1]
        suppress_labels = np.where(
            blob_sums / (blob_sums[np.argsort(blob_sums)[::-1][1]]) < suppress_label_thresh
        )[0]
        for sl in suppress_labels:
            coronal_brain_mask_small_holes_closed = np.where(
                blobs_before_processing == sl, 1, coronal_brain_mask_small_holes_closed
            )
            coronal_bin_closed = np.where(
                blobs_before_processing == sl, 1, coronal_bin_closed
            )
    except Exception:
        coronal_brain_mask_small_holes_closed = coronal_brain_mask
        coronal_bin_closed = coronal_bin

    # Step 7: Morphological open with asymmetric kernel → break choroid plexus
    cp_break_kernel = np.array([[1, 1, 1], [0, 1, 0], [1, 1, 1]], "uint8")
    open_coronal_brain_mask = cv.morphologyEx(
        np.uint8(coronal_brain_mask_small_holes_closed),
        cv.MORPH_OPEN,
        cp_break_kernel,
        iterations=coronal_close_num_iterations,
    )

    # Step 8: Ventricles = union of Otsu-dark and adaptive-dark regions inside brain
    ventricle_mask_from_adaptive_0 = np.where(
        full_brain_mask - open_coronal_brain_mask > 0, 1, 0
    )
    ventricle_mask_from_otsu_0 = np.where(
        (1 - coronal_bin) * full_brain_mask > 0, 1, 0
    )
    ventricles = np.logical_or(ventricle_mask_from_otsu_0, ventricle_mask_from_adaptive_0)

    if visualize:
        plt.imshow(coronal_plane, cmap="gray")
        plt.title("Coronal plane (raw MRI, no windowing)")
        plt.show()
        plt.imshow(coronal_gauss_blurred, cmap="gray")
        plt.title("Coronal gauss blurred")
        plt.show()
        plt.imshow(coronal_bin, cmap="gray")
        plt.title("Coronal Otsu")
        plt.show()
        plt.imshow(full_brain_mask, cmap="gray")
        plt.title("Full brain mask")
        plt.show()
        plt.imshow(coronal_brain_mask, cmap="gray")
        plt.title("Adaptive coronal brain mask")
        plt.show()
        plt.imshow(coronal_brain_mask_small_holes_closed, cmap="gray")
        plt.title("After suppress labels - adaptive")
        plt.show()
        plt.imshow(coronal_bin_closed, cmap="gray")
        plt.title("After suppress labels - Otsu")
        plt.show()
        plt.imshow(open_coronal_brain_mask, cmap="gray")
        plt.title("Adaptive coronal brain mask - Open (choroid plexus removed)")
        plt.show()
        plt.imshow(ventricles, cmap="gray")
        plt.title("Segmented lateral ventricles")
        plt.show()

    return ventricles, full_brain_mask, cor_pc_index, coronal_plane

## Ventricle processing: `process_ventricles()`

Takes the raw ventricle mask from `segment_ventricles()` and refines it:

1. **Mask above PC** — zeros out everything at or below the PC row index (inferior horn of lateral ventricles is above PC in image coordinates).
2. **Middle ROI** — creates a small rectangular region of interest centred on the brain centroid. This is unioned with the above-PC mask to keep the central ventricular body even when isolated.
3. **Largest connected component** — removes stray blobs, keeping only the dominant ventricular mass.
4. **Percentile crop** — removes the topmost portion based on the 75th percentile of ventricle row positions, cutting off superior atrophy artefacts.
5. **Dynamic median smoothing** — applies median blur; if the ventricle-to-brain ratio drops below `ventricle_to_brain_ratio_thresh`, rolls back the smoothing kernel by 2 pixels (up to 4 times).

In [ ]:
def process_ventricles(ventricles, cor_pc_index, full_brain_mask, params, visualize=False):
    """Refines the raw ventricle mask: masks above PC, ROI, CC, percentile crop, dynamic smoothing.

    Ported directly from CT MaxEccLV process_venticles (note original spelling).

    Parameters
    ----------
    ventricles : np.ndarray
        Raw 2D binary ventricle mask from segment_ventricles().
    cor_pc_index : int
        Coronal slice Y-index used to extract the plane (row = PC level).
    full_brain_mask : np.ndarray
        Full brain binary mask (2D).
    params : dict
        Processing parameters dict (ventricles_processing_smoothing_params).
    visualize : bool, optional
        If True, displays intermediate steps (default False).

    Returns
    -------
    ventricles_smoothed : np.ndarray
        Smoothed, cleaned binary ventricle mask (2D, uint8).
    """
    middle_roi_upper_dec = params["middle_roi_upper_dec"]
    middle_roi_lower_inc = params["middle_roi_lower_inc"]
    middle_roi_left_dec = params["middle_roi_left_dec"]
    middle_roi_right_dec = params["middle_roi_right_dec"]
    scale_upper_thresh = params["scale_upper_thresh"]
    smoothing_factor = params["smoothing_factor"]
    dec_thresh = params["dec_thresh"]
    ventricle_to_brain_ratio_thresh = params["ventricle_to_brain_ratio_thresh"]

    # Brain centroid
    M = cv.moments(np.uint8(full_brain_mask))
    if M["m00"] != 0:
        bcX = int(M["m10"] / M["m00"])
        bcY = int(M["m01"] / M["m00"])
    else:
        bcX, bcY = full_brain_mask.shape[1] // 2, full_brain_mask.shape[0] // 2

    # Step 1: Mask ventricles above PC
    # In image coordinates (row=0 at top), ventricles above PC have row < cor_pc_index
    ventricles_above_pc = ventricles.copy()
    ventricles_above_pc[cor_pc_index:, :] = 0

    # Step 2: Middle ROI (small rectangle around brain centroid)
    middle_roi = np.zeros(ventricles.shape)
    middle_roi[
        bcY - middle_roi_upper_dec : bcY + middle_roi_lower_inc,
        bcX - middle_roi_left_dec : bcX + middle_roi_right_dec,
    ] = 1

    clean_ven_1 = np.logical_or(middle_roi, ventricles_above_pc)

    # Step 3: Keep largest connected component
    blobs_labels = measure.label(clean_ven_1, background=0)
    largest_cc_ven = getLargestCC(blobs_labels)
    [row_v, col_v] = (1 - largest_cc_ven).nonzero()
    ventricles_clean_above_pc = ventricles_above_pc.copy()
    ventricles_clean_above_pc[row_v, col_v] = 0

    [vr, vc] = ventricles_clean_above_pc.nonzero()
    if len(vr) == 0:
        print("Warning: ventricle mask is empty above PC. Returning crude LV mask above PC.")
        return ventricles_clean_above_pc

    # Step 4: Percentile crop — remove topmost artefacts
    lv_roi = np.zeros(ventricles_clean_above_pc.shape)
    iqr_row = np.percentile(vr, 75) - np.percentile(vr, 25)
    upper_thresh_row = int(np.round(np.percentile(vr, 75) + scale_upper_thresh * iqr_row))
    lv_roi[:upper_thresh_row, :] = 1
    ventricles_clean_above_pc_1 = np.uint8(lv_roi * ventricles_clean_above_pc)

    if visualize:
        plt.imshow(ventricles, cmap="gray")
        plt.imshow(middle_roi, alpha=0.2)
        plt.title("Ventricles with middle ROI shown")
        plt.show()
        plt.imshow(ventricles_clean_above_pc, cmap="gray")
        plt.title("Ventricles cleaned above PC")
        plt.show()
        plt.imshow(ventricles_clean_above_pc_1, cmap="gray")
        plt.title("Ventricles cleaned above PC, only until 75th percentile")
        plt.show()

    # Step 5: Dynamic median smoothing
    while True:
        ventricles_smoothed = cv.medianBlur(
            np.uint8(ventricles_clean_above_pc_1), smoothing_factor - dec_thresh
        )
        if visualize:
            plt.imshow(ventricles_smoothed, cmap="gray")
            plt.title(f"Ventricles smoothed at {smoothing_factor - dec_thresh}")
            plt.show()
        if np.sum(ventricles_smoothed) / np.sum(full_brain_mask) >= ventricle_to_brain_ratio_thresh:
            break
        else:
            print("Smoothing factor being reduced")
            dec_thresh = dec_thresh + 2
            if dec_thresh > 4:
                print("Ventricles too small")
                break

    if visualize:
        plt.imshow(ventricles_smoothed, cmap="gray")
        plt.title("Final ventricles smoothed")
        plt.show()

    return ventricles_smoothed

## Connected component analysis: `num_connected_comps_cv()`

Determines whether the left and right lateral ventricles appear as a single connected mass (typical in mild hydrocephalus) or as two disconnected blobs (typical in severe NPH or normal anatomy).

Returns `(disconnected_case, lat_ventricles)` where `disconnected_case` is a boolean and `lat_ventricles` is the cleaned ventricle mask with spurious blobs removed.

If exactly 3 labels (background + 2 blobs) are present and both blobs are symmetric around the image midline and neither is a non-ventricular mass, the case is flagged as disconnected. Otherwise, small off-centre blobs are removed.

In [ ]:
def num_connected_comps_cv(
    clean_ventricles,
    suppress_blob_area_ratio=0.3,
    disconnectedLV_height_diff_tol=5,
    disconnectedLV_area_diff_tol=0.05,
    disconnectedLV_x_diff_tol=20,
    area_check_thresh=0.5,
    connected_case_blob_ratio=0.5,
    visualize=False,
):
    """Evaluates the cleaned ventricular mask to determine if LV are connected or disconnected.

    Parameters
    ----------
    clean_ventricles : np.ndarray
        Smoothed 2D binary ventricle mask from process_ventricles().
    suppress_blob_area_ratio : float
        Blobs smaller than this fraction of the largest blob are removed.
    disconnectedLV_height_diff_tol : int
        Tolerance in rows for centroid height difference when calling disconnected.
    disconnectedLV_area_diff_tol : float
        Tolerance for blob area ratio when calling disconnected.
    disconnectedLV_x_diff_tol : int
        Maximum centroid X distance from image midline for disconnected case.
    area_check_thresh : float
        Area ratio threshold below which the smaller blob is not a false CC.
    connected_case_blob_ratio : float
        If larger blob is much bigger than smaller, treat as connected (one-sided).
    visualize : bool, optional
        If True, displays output mask (default False).

    Returns
    -------
    disconnected_case : bool
        True if two separate lateral ventricle blobs are detected.
    lat_ventricles : np.ndarray
        Cleaned binary ventricle mask.
    """
    processed_blobs = measure.label(clean_ventricles, background=0)

    if len(np.unique(processed_blobs)) <= 1:
        return False, np.where(processed_blobs > 0, 1, 0)

    # Remove blobs smaller than suppress_blob_area_ratio * largest blob
    counts = np.bincount(processed_blobs.flatten())
    counts[0] = 0  # zero out background
    largest_blob_count = counts.max()
    for label_idx in range(1, len(counts)):
        if counts[label_idx] > 0 and counts[label_idx] < suppress_blob_area_ratio * largest_blob_count:
            processed_blobs = np.where(processed_blobs == label_idx, 0, processed_blobs)

    # Re-label after suppression
    rem_labels = np.unique(processed_blobs)
    new_labels = range(len(rem_labels))
    map_dict = {r: n for r, n in zip(rem_labels, new_labels)}
    new_blobs = np.zeros_like(processed_blobs)
    for k, v in map_dict.items():
        new_blobs[processed_blobs == k] = v
    processed_blobs = new_blobs

    [pr, pc] = processed_blobs.nonzero()
    if len(pc) == 0:
        return False, np.where(processed_blobs > 0, 1, 0)

    im_cen = (max(pc) + min(pc)) // 2
    disconnected_case = False

    if len(np.unique(processed_blobs)) == 3:  # background + 2 blobs
        counts = np.bincount(processed_blobs.flatten())

        smallest_blob_arg = np.argsort(counts)[0]
        smallest_blob = np.where(processed_blobs == smallest_blob_arg, 1, 0)
        [sb_r, sb_c] = smallest_blob.nonzero()
        smallest_blob_height = (min(sb_r) + max(sb_r)) // 2 if len(sb_r) > 0 else 0
        smallest_blob_cen_x = (min(sb_c) + max(sb_c)) // 2 if len(sb_c) > 0 else 0

        mid_blob_arg = np.argsort(counts)[1]
        mid_blob = np.where(processed_blobs == mid_blob_arg, 1, 0)
        [mb_r, mb_c] = mid_blob.nonzero()
        mid_blob_height = (min(mb_r) + max(mb_r)) // 2 if len(mb_r) > 0 else 0
        mid_blob_cen_x = (min(mb_c) + max(mb_c)) // 2 if len(mb_c) > 0 else 0

        nv_mass = (smallest_blob_height < min(mb_r)) | (smallest_blob_height > max(mb_r))
        area_check = (
            np.sum(mid_blob) - np.sum(smallest_blob)
            > area_check_thresh * np.sum(mid_blob)
        )

        if (
            (np.abs(smallest_blob_cen_x - im_cen) <= disconnectedLV_x_diff_tol)
            and (np.abs(mid_blob_cen_x - im_cen) <= disconnectedLV_x_diff_tol)
            and (not nv_mass)
            and (not area_check)
        ):
            disconnected_case = True
        else:
            processed_blobs = np.where(
                processed_blobs == smallest_blob_arg, 0, processed_blobs
            )

    processed_blobs = np.where(processed_blobs > 0, 1, 0)

    if visualize:
        plt.imshow(processed_blobs, cmap="gray")
        plt.title("Output of connected component analysis")
        plt.show()

    return disconnected_case, processed_blobs

## Disconnected case handling: `check_if_disconnected_case_too_small()`

When `num_connected_comps_cv()` returns `disconnected_case=True`, this function validates whether both disconnected blobs are large enough to process independently, or if one is so small (or asymmetric) that the case should be collapsed back to a single-blob connected scenario.

Also partitions the mask into `left_ven_con` and `right_ven_con` (left and right halves of the ventricle mask, split at the image midline) for downstream contour tracing.

In [ ]:
def check_if_disconnected_case_too_small(
    lat_ventricles,
    full_brain_mask,
    disconnected_case,
    process_left,
    process_right,
    lv_to_full_brain_thresh=0.004,
    connected_case_blob_ratio=0.5,
    visualize=False,
):
    """Validates disconnected ventricle blobs and partitions into left/right halves.

    Parameters
    ----------
    lat_ventricles : np.ndarray
        Binary 2D ventricle mask (output of num_connected_comps_cv).
    full_brain_mask : np.ndarray
        Binary 2D full brain mask.
    disconnected_case : bool
        Flag from num_connected_comps_cv indicating two separate blobs.
    process_left : int
        1 if the left ventricle should be processed, 0 otherwise.
    process_right : int
        1 if the right ventricle should be processed, 0 otherwise.
    lv_to_full_brain_thresh : float
        Minimum ratio of ventricle area to brain area to process both sides.
    connected_case_blob_ratio : float
        If one blob is much larger, collapse to connected single-blob case.
    visualize : bool, optional
        If True, displays intermediate steps (default False).

    Returns
    -------
    lat_ventricles : np.ndarray
        Possibly updated binary ventricle mask.
    disconnected_case : bool
        Possibly updated disconnected flag.
    left_ven_con : np.ndarray
        Left-half ventricle mask.
    right_ven_con : np.ndarray
        Right-half ventricle mask.
    process_left : int
        Updated flag (0 or 1).
    process_right : int
        Updated flag (0 or 1).
    """
    blob_labels = measure.label(lat_ventricles, background=0)

    [r, c] = lat_ventricles.nonzero()
    if len(c) == 0:
        left_ven_con = np.zeros(lat_ventricles.shape)
        right_ven_con = np.zeros(lat_ventricles.shape)
        return lat_ventricles, disconnected_case, left_ven_con, right_ven_con, 0, 0

    midpoint_c = (min(c) + max(c)) // 2

    # Partition into left / right halves
    left_ven_con = lat_ventricles.copy()
    left_ven_con[:, midpoint_c:] = 0

    right_ven_con = lat_ventricles.copy()
    right_ven_con[:, :midpoint_c] = 0

    # If both blobs together are too small relative to brain, check if one dominates
    if (
        (len(np.unique(blob_labels)) > 2)
        and (np.sum(lat_ventricles) / np.sum(full_brain_mask) < lv_to_full_brain_thresh)
    ):
        counts = np.bincount(blob_labels.flatten())
        counts[0] = 0
        largest_2_indices = np.argsort(counts)[::-1][:2]

        larger_blob = np.where(blob_labels == largest_2_indices[0], 1, 0)
        [lbr, lbc] = larger_blob.nonzero()
        lbc_m = (min(lbc) + max(lbc)) // 2 if len(lbc) > 0 else 0

        smaller_blob = np.where(blob_labels == largest_2_indices[1], 1, 0)
        [sbr, sbc] = smaller_blob.nonzero()
        sbc_m = (min(sbc) + max(sbc)) // 2 if len(sbc) > 0 else 0

        blob_sums = [np.sum(smaller_blob), np.sum(larger_blob)]

        if blob_sums[1] > 0 and np.abs(
            (blob_sums[1] - blob_sums[0]) / blob_sums[1]
        ) > connected_case_blob_ratio:
            lat_ventricles = np.where(blob_labels == largest_2_indices[0], 1, 0)
            disconnected_case = False

        # Determine which side is missing
        if sbc_m < lbc_m:
            process_left = 0
        else:
            process_right = 0

        if visualize:
            plt.imshow(lat_ventricles, cmap="gray")
            plt.title("Disconnected case: processed as single blob")
            plt.show()

    return lat_ventricles, disconnected_case, left_ven_con, right_ven_con, process_left, process_right

## Contour tracing: `trace_contour()`

Traces the **superior (upper) boundary** of the left and right lateral ventricles on the coronal plane. The traced boundary points are used for ellipse fitting.

**Direction priority for left ventricle (rightward traversal):** NW → N → NE → E → SE → S
**Direction priority for right ventricle (leftward traversal):** NE → N → NW → W → SW → S

These differ from the CallosalAngle version (`trace_left_contour` / `trace_right_contour`), which trace the **inferior** medial wall moving SE/SW. Here we trace the superior roof moving toward the midline, collecting points for the ellipse semi-major axis.

After tracing, points are filtered to exclude the medial 20% x-range to avoid midline artefacts. Returns `(left_con_cand_pts, right_con_cand_pts, left_pts, right_pts)`.

In [ ]:
def trace_contour(
    ven_contour,
    left_ven_con,
    right_ven_con,
    process_left,
    process_right,
    disconnected_case,
    cX,
    cY,
    max_iters=10000,
):
    """Traces the superior boundary of the left and right lateral ventricles.

    Used to collect contour points for ellipse fitting (MaxEccLV). Direction
    priority: NW/N/NE/E/SE/S for left ventricle; NE/N/NW/W/SW/S for right.
    This differs from the CallosalAngle contour tracing which traces the
    inferior medial wall moving SE/SW.

    Parameters
    ----------
    ven_contour : np.ndarray
        2D binary contour mask of the ventricles (output of moore_contour).
    left_ven_con : np.ndarray or int
        Left-half ventricle mask (from check_if_disconnected_case_too_small),
        or -1 in the connected case.
    right_ven_con : np.ndarray or int
        Right-half ventricle mask, or -1 in the connected case.
    process_left : int
        1 to trace left ventricle, 0 to skip.
    process_right : int
        1 to trace right ventricle, 0 to skip.
    disconnected_case : bool
        True if ventricles appear as two separate blobs.
    cX : int
        X coordinate of the overall ventricle mass centroid.
    cY : int
        Y coordinate of the overall ventricle mass centroid.
    max_iters : int, optional
        Safety limit on tracing iterations (default 10000).

    Returns
    -------
    left_con_cand_pts : list of (int, int)
        Filtered (x, y) candidate contour points for the left ventricle.
    right_con_cand_pts : list of (int, int)
        Filtered (x, y) candidate contour points for the right ventricle.
    left_pts : list of (int, int)
        All traced left ventricle contour points (unfiltered).
    right_pts : list of (int, int)
        All traced right ventricle contour points (unfiltered).
    """
    max_row, max_col = ven_contour.shape
    [r, c] = ven_contour.nonzero()
    if len(c) == 0:
        return [], [], [], []

    midpoint_c = (min(c) + max(c)) // 2

    left_pts = []
    right_pts = []

    # ==========================================
    # Left ventricle tracing
    # ==========================================
    if process_left == 1:
        if not disconnected_case:
            [lvc_r, lvc_c] = ven_contour[:, :midpoint_c].nonzero()
            mask_to_check = ven_contour

            try:
                lv_tp_x = min(lvc_c[lvc_r == cY])
            except Exception:
                lv_tp_x = min(lvc_c) if len(lvc_c) > 0 else midpoint_c // 2

            stop_row_l = max(lvc_r[lvc_c == max(lvc_c)]) if len(lvc_c) > 0 else cY
            stop_col_l = max(lvc_c) - 2 if len(lvc_c) > 0 else midpoint_c

            # Connected case: stop at centroid row or at medial terminus
            stop_condition = lambda r, c: (r <= cY) and not (r == stop_row_l and c == stop_col_l)
        else:
            [lvc_r, lvc_c] = left_ven_con.nonzero()
            mask_to_check = left_ven_con

            try:
                lv_tp_x = min(lvc_c[lvc_r == cY])
            except Exception:
                lv_tp_x = min(lvc_c) if len(lvc_c) > 0 else midpoint_c // 2

            stop_row_l = max(lvc_r[lvc_c == max(lvc_c)]) if len(lvc_c) > 0 else cY
            stop_col_l = max(lvc_c) - 2 if len(lvc_c) > 0 else midpoint_c

            stop_condition = lambda r, c: (r <= cY) and not (r == stop_row_l and c == stop_col_l)

        row, col = cY, lv_tp_x
        if mask_to_check[row, col] == 0 and len(lvc_r[lvc_c == col]) > 0:
            row = max(lvc_r[lvc_c == col])

        north_west_count = 0
        south_step = 0
        iters = 0

        while stop_condition(row, col):
            if iters > max_iters or row < 1 or row >= max_row - 1 or col < 1 or col >= max_col - 1:
                break
            iters += 1

            if (ven_contour[row - 1, col - 1] > 0) and (north_west_count <= 10):  # NW
                left_pts.append((col - 1, row - 1))
                row, col = row - 1, col - 1
                north_west_count += 1
                south_step = 0
            elif (ven_contour[row - 1, col] > 0) and (south_step == 0):  # N
                left_pts.append((col, row - 1))
                row, col = row - 1, col
                south_step = 0
            elif ven_contour[row - 1, col + 1] > 0:  # NE
                left_pts.append((col + 1, row - 1))
                row, col = row - 1, col + 1
                south_step = 0
            elif ven_contour[row, col + 1] > 0:  # E
                left_pts.append((col + 1, row))
                row, col = row, col + 1
                south_step = 0
            elif ven_contour[row + 1, col + 1] > 0:  # SE
                left_pts.append((col + 1, row + 1))
                row, col = row + 1, col + 1
                south_step = 0
            else:  # S
                left_pts.append((col, row + 1))
                row, col = row + 1, col
                south_step = 1

    # ==========================================
    # Right ventricle tracing
    # ==========================================
    if process_right == 1:
        if not disconnected_case:
            [rvc_r, rvc_c] = ven_contour[:, midpoint_c + 1:].nonzero()
            mask_to_check = ven_contour

            try:
                rv_tp_x = max(rvc_c[rvc_r == cY]) + midpoint_c
            except Exception:
                rv_tp_x = max(rvc_c) + midpoint_c if len(rvc_c) > 0 else midpoint_c + midpoint_c // 2

            stop_condition = lambda r, c: (c > cX) and (r <= cY)

            row, col = cY, rv_tp_x
            if mask_to_check[row, col] == 0 and len(rvc_r[rvc_c == max(rvc_c)]) > 0:
                row = max(rvc_r[rvc_c == max(rvc_c)])
        else:
            [rvc_r, rvc_c] = right_ven_con.nonzero()
            mask_to_check = right_ven_con

            try:
                rv_tp_x = max(rvc_c[rvc_r == cY])
            except Exception:
                rv_tp_x = max(rvc_c) if len(rvc_c) > 0 else midpoint_c + midpoint_c // 2

            stop_row_r = max(rvc_r[rvc_c == min(rvc_c)]) if len(rvc_c) > 0 else cY
            stop_col_r = min(rvc_c) + 2 if len(rvc_c) > 0 else midpoint_c

            stop_condition = lambda r, c: (r <= cY) and not (r == stop_row_r and c == stop_col_r)

            row, col = cY, rv_tp_x
            if mask_to_check[row, col] == 0 and len(rvc_r[rvc_c == col]) > 0:
                row = max(rvc_r[rvc_c == col])

        north_east_count = 0
        south_step = 0
        iters = 0

        while stop_condition(row, col):
            if iters > max_iters or row < 1 or row >= max_row - 1 or col < 1 or col >= max_col - 1:
                break
            iters += 1

            if (ven_contour[row - 1, col + 1] > 0) and (north_east_count <= 10):  # NE
                right_pts.append((col + 1, row - 1))
                row, col = row - 1, col + 1
                north_east_count += 1
                south_step = 0
            elif (ven_contour[row - 1, col] > 0) and (south_step == 0):  # N
                right_pts.append((col, row - 1))
                row, col = row - 1, col
                south_step = 0
            elif ven_contour[row - 1, col - 1] > 0:  # NW
                right_pts.append((col - 1, row - 1))
                row, col = row - 1, col - 1
                south_step = 0
            elif ven_contour[row, col - 1] > 0:  # W
                right_pts.append((col - 1, row))
                row, col = row, col - 1
                south_step = 0
            elif ven_contour[row + 1, col - 1] > 0:  # SW
                right_pts.append((col - 1, row + 1))
                row, col = row + 1, col - 1
                south_step = 0
            else:  # S
                right_pts.append((col, row + 1))
                row, col = row + 1, col
                south_step = 1

    # ==========================================
    # Post-processing: filter medial 20% x-range
    # ==========================================
    left_pts = list(set(left_pts))
    right_pts = list(set(right_pts))

    left_con_cand_pts = []
    if left_pts:
        try:
            left_min_col = min(x for x, y in left_pts)
            left_max_col = max(x for x, y in left_pts)
            left_x_range = left_max_col - left_min_col
            left_low_thresh = left_max_col - 0.2 * left_x_range
            left_con_cand_pts = [(x, y) for x, y in left_pts if x <= left_low_thresh]
        except Exception:
            left_con_cand_pts = left_pts

    right_con_cand_pts = []
    if right_pts:
        try:
            right_max_col = max(x for x, y in right_pts)
            right_min_col = min(x for x, y in right_pts)
            right_x_range = right_max_col - right_min_col
            right_low_thresh = right_min_col + 0.2 * right_x_range
            right_con_cand_pts = [(x, y) for x, y in right_pts if x >= right_low_thresh]
        except Exception:
            right_con_cand_pts = right_pts

    return left_con_cand_pts, right_con_cand_pts, left_pts, right_pts

## Mirror points and ellipse fitting

### `append_mirror_pts()`

The superior contour (traced by `trace_contour`) is clean because it follows the outer roof of the ventricle. The inferior boundary is noisy due to choroid plexus and septal irregularities. To obtain a complete ellipse without relying on the noisy inferior boundary, we **mirror** the superior contour points across each ventricle's centroid. This synthesizes a symmetric inferior contour that, combined with the superior points, gives a full ellipse worth of data for least-squares fitting.

### `calculate_max_ecc()`

Fits an ellipse to the combined (superior + mirrored inferior) contour points using least-squares.

**Ellipse equation (centred form):**
$$A(x - c_x)^2 + B(y - c_y)^2 + C(x - c_x)(y - c_y) = 1$$

**Normal equations:**
$$\text{coefs} = (X^T X)^{-1} X^T Y, \quad X_i = [(x_i - c_x)^2,\ (y_i - c_y)^2,\ (x_i - c_x)(y_i - c_y)], \quad Y_i = 1$$

**Semi-axes and rotation:**
$$\alpha = \frac{1}{2}\arctan\!\left(\frac{C - A + B}{A - B}\right) + \frac{1}{2}, \quad
b = \sqrt{\frac{2}{A + B - (C - A + B) / ((\tan 2\alpha - 1)\cos 2\alpha)}}, \quad
a = \sqrt{\frac{1}{A + B - 1/b^2}}$$

**Eccentricity** (assuming $a \geq b$): $e = \sqrt{1 - b^2/a^2}$

Returns `max(ecc_L, ecc_R)`. A value of -100 indicates a failed fit for that side.

In [ ]:
def append_mirror_pts(lat_ventricles, left_con_cand_pts, right_con_cand_pts):
    """Reflects superior contour points across ventricular centroids to synthesize inferior contour.

    The choroid plexus and septal irregularities make the inferior ventricle
    boundary noisy. Rather than tracing it directly, this reflects the clean
    superior contour across each ventricle's centroid to produce a synthetic
    inferior boundary suitable for ellipse fitting.

    Parameters
    ----------
    lat_ventricles : np.ndarray
        Binary mask of the segmented lateral ventricles (2D).
    left_con_cand_pts : list of (int, int)
        Filtered (x, y) superior contour points for the left ventricle.
    right_con_cand_pts : list of (int, int)
        Filtered (x, y) superior contour points for the right ventricle.

    Returns
    -------
    left_pts_inv : list of (int, int)
        Synthesized inferior left ventricle contour points.
    right_pts_inv : list of (int, int)
        Synthesized inferior right ventricle contour points.
    """
    [r, c] = lat_ventricles.nonzero()

    if len(c) == 0:
        print("Warning: lat_ventricles mask is empty. Cannot compute mirror points.")
        return [], []

    midpoint_c = (min(c) + max(c)) // 2

    left_pts_inv = []
    if len(left_con_cand_pts) > 0:
        left_half = np.uint8(lat_ventricles[:, :midpoint_c])
        lv_M = cv.moments(left_half)
        if lv_M["m00"] != 0:
            cX_L = int(lv_M["m10"] / lv_M["m00"])
            cY_L = int(lv_M["m01"] / lv_M["m00"])
            # Reflect: (x, y) -> (2*cx - x, 2*cy - y)
            left_pts_inv = [(2 * cX_L - x, 2 * cY_L - y) for x, y in left_con_cand_pts]

    right_pts_inv = []
    if len(right_con_cand_pts) > 0:
        right_half = np.uint8(lat_ventricles[:, midpoint_c:])
        rv_M = cv.moments(right_half)
        if rv_M["m00"] != 0:
            cX_R = int(rv_M["m10"] / rv_M["m00"]) + midpoint_c
            cY_R = int(rv_M["m01"] / rv_M["m00"])
            right_pts_inv = [(2 * cX_R - x, 2 * cY_R - y) for x, y in right_con_cand_pts]

    return left_pts_inv, right_pts_inv


def calculate_max_ecc(coronal_plane, lat_ventricles, left_con_cand_pts, right_con_cand_pts):
    """Fits ellipses to L/R ventricle contours and returns the maximum eccentricity.

    Uses least-squares to fit A(x-cx)^2 + B(y-cy)^2 + C(x-cx)(y-cy) = 1.
    Extracts semi-axes a, b and rotation alpha from the fitted coefficients.
    Eccentricity = sqrt(1 - (min_axis/max_axis)^2).

    Parameters
    ----------
    coronal_plane : np.ndarray
        Raw coronal MRI slice (2D, for visualization only).
    lat_ventricles : np.ndarray
        Binary 2D ventricle mask.
    left_con_cand_pts : list of (int, int)
        Combined superior + mirrored inferior contour points for left ventricle.
    right_con_cand_pts : list of (int, int)
        Combined superior + mirrored inferior contour points for right ventricle.

    Returns
    -------
    max_ecc : float
        Maximum eccentricity across left and right ventricles.
        Returns -100 if both fits fail.
    """
    [r, c] = lat_ventricles.nonzero()
    midpoint_c = (min(c) + max(c)) // 2 if len(c) > 0 else lat_ventricles.shape[1] // 2

    left_plot_flag = 1
    right_plot_flag = 1

    # ==========================================
    # Left ventricle
    # ==========================================
    if len(left_con_cand_pts) > 0:
        try:
            lv_M = cv.moments(np.uint8(lat_ventricles[:, :midpoint_c]))
            cX_L = int(lv_M["m10"] / lv_M["m00"])
            cY_L = int(lv_M["m01"] / lv_M["m00"])

            X_L = np.array(
                [((x - cX_L) ** 2, (y - cY_L) ** 2, (x - cX_L) * (y - cY_L))
                 for x, y in left_con_cand_pts]
            ).reshape(len(left_con_cand_pts), 3)
            Y_L = np.ones((len(left_con_cand_pts), 1))

            coefs_L = np.matmul(
                np.linalg.inv(np.matmul(X_L.T, X_L)),
                np.matmul(X_L.T, Y_L),
            )

            alpha_L = np.arctan(
                ((coefs_L[2] - coefs_L[0] + coefs_L[1]) / (coefs_L[0] - coefs_L[1])) + 1
            ) / 2
            b_L = np.sqrt(
                abs(
                    2 / (
                        coefs_L[0] + coefs_L[1]
                        - (coefs_L[2] - coefs_L[0] + coefs_L[1])
                        / ((np.tan(2 * alpha_L) - 1) * np.cos(2 * alpha_L))
                    )
                )
            )
            a_L = np.sqrt(abs(1 / (coefs_L[0] + coefs_L[1] - 1 / b_L ** 2)))

            left_el = np.zeros(lat_ventricles.shape)
            for y in range(left_el.shape[0]):
                for x in range(left_el.shape[1]):
                    f = (
                        (((x - cX_L) * np.cos(alpha_L) + (y - cY_L) * np.sin(alpha_L)) / a_L) ** 2
                        + (((x - cX_L) * np.sin(alpha_L) - (y - cY_L) * np.cos(alpha_L)) / b_L) ** 2
                    )
                    if f <= 1:
                        left_el[y, x] = 1

            if a_L >= b_L:
                ecc_L = np.sqrt(1 - (b_L ** 2) / (a_L ** 2))[0]
            else:
                ecc_L = np.sqrt(1 - (a_L ** 2) / (b_L ** 2))[0]

        except Exception:
            ecc_L = -100
            left_plot_flag = 0
    else:
        ecc_L = -100
        left_plot_flag = 0

    # ==========================================
    # Right ventricle
    # ==========================================
    if len(right_con_cand_pts) > 0:
        try:
            rv_M = cv.moments(np.uint8(lat_ventricles[:, midpoint_c:]))
            cX_R = int(rv_M["m10"] / rv_M["m00"]) + midpoint_c
            cY_R = int(rv_M["m01"] / rv_M["m00"])

            X_R = np.array(
                [((x - cX_R) ** 2, (y - cY_R) ** 2, (x - cX_R) * (y - cY_R))
                 for x, y in right_con_cand_pts]
            ).reshape(len(right_con_cand_pts), 3)
            Y_R = np.ones((len(right_con_cand_pts), 1))

            coefs_R = np.matmul(
                np.linalg.inv(np.matmul(X_R.T, X_R)),
                np.matmul(X_R.T, Y_R),
            )

            alpha_R = np.arctan(
                ((coefs_R[2] - coefs_R[0] + coefs_R[1]) / (coefs_R[0] - coefs_R[1])) + 1
            ) / 2
            b_R = np.sqrt(
                abs(
                    2 / (
                        coefs_R[0] + coefs_R[1]
                        - (coefs_R[2] - coefs_R[0] + coefs_R[1])
                        / ((np.tan(2 * alpha_R) - 1) * np.cos(2 * alpha_R))
                    )
                )
            )
            a_R = np.sqrt(abs(1 / (coefs_R[0] + coefs_R[1] - 1 / b_R ** 2)))

            right_el = np.zeros(lat_ventricles.shape)
            for y in range(right_el.shape[0]):
                for x in range(right_el.shape[1]):
                    f = (
                        (((x - cX_R) * np.cos(alpha_R) + (y - cY_R) * np.sin(alpha_R)) / a_R) ** 2
                        + (((x - cX_R) * np.sin(alpha_R) - (y - cY_R) * np.cos(alpha_R)) / b_R) ** 2
                    )
                    if f <= 1:
                        right_el[y, x] = 1

            if a_R >= b_R:
                ecc_R = np.sqrt(1 - (b_R ** 2) / (a_R ** 2))[0]
            else:
                ecc_R = np.sqrt(1 - (a_R ** 2) / (b_R ** 2))[0]

        except Exception:
            ecc_R = -100
            right_plot_flag = 0
    else:
        ecc_R = -100
        right_plot_flag = 0

    # ==========================================
    # Visualization
    # ==========================================
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(coronal_plane, cmap="gray")
    if left_plot_flag == 1:
        [ax.scatter(x, y, marker="x", color="w", s=5) for (x, y) in left_con_cand_pts]
    if right_plot_flag == 1:
        [ax.scatter(x, y, marker="x", color="w", s=5) for (x, y) in right_con_cand_pts]
    ax.set_title(f"Ellipse contour points | ecc_L={ecc_L:.3f}  ecc_R={ecc_R:.3f}")
    plt.show()

    return max(ecc_L, ecc_R)

## Load PC coordinates

For OASIS-1 the Posterior Commissure has a known fixed voxel coordinate in the atlas space (`OASIS_PC_VOXEL = (85, 87, 73)`). For ABCD/custom datasets, per-subject AC/PC coordinates are loaded from `.txt` files produced by Charlie's ART alignment script.

In [ ]:
# For OASIS, PC is at a fixed atlas voxel; for ABCD/custom, load from per-subject txt files.
if ACPC_MODE == "oasis_atlas":
    # OASIS_PC_VOXEL = (x_vox, y_vox, z_vox) = (85, 87, 73) in atlas space.
    # For coronal slicing of img_arr (shape Z, Y, X), the Y-index is OASIS_PC_VOXEL[1].
    PC_Y_INDEX_DEFAULT = OASIS_PC_VOXEL[1]  # 87
    print(f"OASIS atlas PC voxel: {OASIS_PC_VOXEL}")
    print(f"Coronal Y index to use: {PC_Y_INDEX_DEFAULT}")
elif ACPC_MODE == "txt_file":
    # Will be loaded per-subject in the processing loop
    PC_Y_INDEX_DEFAULT = None
    print("AC/PC loaded per-subject from .txt files")

## Discover available scans

In [ ]:
scan_files = sorted(glob.glob(os.path.join(DATA_DIR, SCAN_GLOB), recursive=True))

scan_df = pd.DataFrame({
    "subject_id": [SUBJECT_ID_FROM_PATH(p) for p in scan_files],
    "scan_path": scan_files,
})
print(f"Found {len(scan_df)} scans in {DATA_DIR}")
scan_df.head()

## Test on a single subject

Runs the full pipeline on one scan with `visualize=True` to allow visual inspection of each processing step before batch processing.

**Expected outputs:**
1. `segment_ventricles` — coronal plane → Otsu → floodfill → adaptive → choroid break → ventricle mask
2. `process_ventricles` — above-PC crop → middle ROI → CC → percentile crop → smooth
3. `num_connected_comps_cv` — connected vs disconnected determination
4. `check_if_disconnected_case_too_small` (if disconnected) — validate and partition L/R
5. `trace_contour` — superior boundary tracing
6. `append_mirror_pts` — synthesize inferior boundary
7. `calculate_max_ecc` — ellipse fit overlay and eccentricity value

In [ ]:
# Pick first subject for testing
test_path = scan_df.iloc[0]["scan_path"]
test_id = scan_df.iloc[0]["subject_id"]
print(f"Testing on: {test_id}")
print(f"Path:       {test_path}")

# Load and rotate volume
img = sitk.ReadImage(test_path)
print(f"Size: {img.GetSize()}  Spacing: {img.GetSpacing()}")

img_rotated = rotate_image(img)
img_arr = sitk.GetArrayFromImage(img_rotated)  # shape (Z, Y, X)
print(f"Array shape: {img_arr.shape}")
print(f"Intensity range: [{img_arr.min():.0f}, {img_arr.max():.0f}]")

# Determine PC Y index
if ACPC_MODE == "oasis_atlas":
    pc_y_index = PC_Y_INDEX_DEFAULT
elif ACPC_MODE == "txt_file":
    # Locate the AC/PC txt file for this subject
    acpc_txt_files = glob.glob(
        os.path.join(DATA_DIR, test_id, "**/*_acpc_coords.txt"), recursive=True
    )
    if acpc_txt_files:
        ac_coords, pc_coords = load_acpc_from_txt(acpc_txt_files[0])
        # Convert PC physical coords to voxel index using the rotated image
        pc_idx = img_rotated.TransformPhysicalPointToContinuousIndex(pc_coords)
        pc_y_index = int(np.round(pc_idx[1]))
        print(f"PC physical coords: {pc_coords}  → voxel Y index: {pc_y_index}")
    else:
        print("WARNING: No ACPC txt file found; using default Y=87")
        pc_y_index = 87

print(f"Coronal slice index (PC level): {pc_y_index}")

In [ ]:
# Step 1: Segment ventricles on the coronal plane at PC level
# MRI adaptation: no window_stack_sitk — Otsu operates on raw T1w intensities
ventricles, full_brain_mask, cor_pc_index, coronal_plane = segment_ventricles(
    img_arr, pc_y_index, coronal_plane_processing_params, visualize=True
)
print(f"Coronal PC index used: {cor_pc_index}")
print(f"Ventricle pixels: {int(ventricles.sum())}  Brain pixels: {int(full_brain_mask.sum())}")

In [ ]:
# Step 2: Process ventricles (mask above PC, ROI, CC, percentile crop, smooth)
ventricles_smoothed = process_ventricles(
    ventricles, cor_pc_index, full_brain_mask,
    ventricles_processing_smoothing_params, visualize=True
)
print(f"Smoothed ventricle pixels: {int(ventricles_smoothed.sum())}")

In [ ]:
# Step 3: Connected component analysis
disconnected_case, lat_ventricles = num_connected_comps_cv(
    ventricles_smoothed, visualize=True
)
print(f"Disconnected case: {disconnected_case}")

# Overall centroid (needed for contour tracing)
M = cv.moments(np.uint8(lat_ventricles))
if M["m00"] != 0:
    cX = int(M["m10"] / M["m00"])
    cY = int(M["m01"] / M["m00"])
else:
    cX, cY = lat_ventricles.shape[1] // 2, lat_ventricles.shape[0] // 2
print(f"Ventricle centroid: cX={cX}, cY={cY}")

In [ ]:
# Step 4: Disconnected case handling (partition into L/R halves)
process_left, process_right = 1, 1

if disconnected_case:
    lat_ventricles, disconnected_case, left_ven_con, right_ven_con, process_left, process_right = (
        check_if_disconnected_case_too_small(
            lat_ventricles, full_brain_mask,
            disconnected_case, process_left, process_right,
            visualize=True,
        )
    )
    # Reject a disconnected half if it contains multiple sub-blobs (artefact)
    left_ven_blobs = measure.label(left_ven_con)
    if len(np.bincount(left_ven_blobs.flatten())) > 2:
        process_left = 0

    right_ven_blobs = measure.label(right_ven_con)
    if len(np.bincount(right_ven_blobs.flatten())) > 2:
        process_right = 0
else:
    left_ven_con, right_ven_con = -1, -1

print(f"process_left={process_left}  process_right={process_right}  disconnected_case={disconnected_case}")

In [ ]:
# Step 5: Trace superior contour of lateral ventricles
ven_contour = moore_contour(lat_ventricles)

left_con_cand_pts, right_con_cand_pts, left_pts, right_pts = trace_contour(
    ven_contour, left_ven_con, right_ven_con,
    process_left, process_right,
    disconnected_case, cX, cY,
)
print(f"Left contour candidate points:  {len(left_con_cand_pts)}")
print(f"Right contour candidate points: {len(right_con_cand_pts)}")

In [ ]:
# Step 6: Mirror superior contour points to synthesize inferior boundary
left_pts_inv, right_pts_inv = append_mirror_pts(
    lat_ventricles, left_con_cand_pts, right_con_cand_pts
)

# Combine superior + mirrored inferior for ellipse fitting
if len(left_con_cand_pts) > 0:
    left_con_cand_pts = left_con_cand_pts + left_pts_inv
if len(right_con_cand_pts) > 0:
    right_con_cand_pts = right_con_cand_pts + right_pts_inv

print(f"Left contour points after mirror:  {len(left_con_cand_pts)}")
print(f"Right contour points after mirror: {len(right_con_cand_pts)}")

In [ ]:
# Step 7: Fit ellipses and compute MaxEccLV
max_ecc = calculate_max_ecc(
    coronal_plane, lat_ventricles, left_con_cand_pts, right_con_cand_pts
)
print(f"\nMaxEccLV for {test_id}: {max_ecc:.4f}")

## Batch processing — all subjects

Runs the full pipeline on all discovered scans with `visualize=False`. Catches and logs exceptions per subject to allow the batch to continue.

In [ ]:
results = []

for _, row in scan_df.iterrows():
    subject_id = row["subject_id"]
    vol_path = row["scan_path"]

    try:
        # Load and rotate
        img = sitk.ReadImage(vol_path)
        img_rotated = rotate_image(img)
        img_arr = sitk.GetArrayFromImage(img_rotated)

        # Determine PC Y index
        if ACPC_MODE == "oasis_atlas":
            pc_y_index = PC_Y_INDEX_DEFAULT
        elif ACPC_MODE == "txt_file":
            acpc_txt_files = glob.glob(
                os.path.join(DATA_DIR, subject_id, "**/*_acpc_coords.txt"), recursive=True
            )
            if not acpc_txt_files:
                raise FileNotFoundError(f"No ACPC txt file for {subject_id}")
            ac_coords, pc_coords = load_acpc_from_txt(acpc_txt_files[0])
            pc_idx = img_rotated.TransformPhysicalPointToContinuousIndex(pc_coords)
            pc_y_index = int(np.round(pc_idx[1]))

        # Segment
        ventricles, full_brain_mask, cor_pc_index, coronal_plane = segment_ventricles(
            img_arr, pc_y_index, coronal_plane_processing_params, visualize=False
        )

        # Process
        ventricles_smoothed = process_ventricles(
            ventricles, cor_pc_index, full_brain_mask,
            ventricles_processing_smoothing_params, visualize=False
        )

        # Connected component analysis
        disconnected_case, lat_ventricles = num_connected_comps_cv(
            ventricles_smoothed, visualize=False
        )

        M = cv.moments(np.uint8(lat_ventricles))
        if M["m00"] != 0:
            cX = int(M["m10"] / M["m00"])
            cY = int(M["m01"] / M["m00"])
        else:
            cX, cY = lat_ventricles.shape[1] // 2, lat_ventricles.shape[0] // 2

        # Disconnected handling
        process_left, process_right = 1, 1
        if disconnected_case:
            lat_ventricles, disconnected_case, left_ven_con, right_ven_con, process_left, process_right = (
                check_if_disconnected_case_too_small(
                    lat_ventricles, full_brain_mask,
                    disconnected_case, process_left, process_right,
                    visualize=False,
                )
            )
            left_ven_blobs = measure.label(left_ven_con)
            if len(np.bincount(left_ven_blobs.flatten())) > 2:
                process_left = 0
            right_ven_blobs = measure.label(right_ven_con)
            if len(np.bincount(right_ven_blobs.flatten())) > 2:
                process_right = 0
        else:
            left_ven_con, right_ven_con = -1, -1

        # Trace contour
        ven_contour = moore_contour(lat_ventricles)
        left_con_cand_pts, right_con_cand_pts, left_pts, right_pts = trace_contour(
            ven_contour, left_ven_con, right_ven_con,
            process_left, process_right,
            disconnected_case, cX, cY,
        )

        # Mirror and combine
        left_pts_inv, right_pts_inv = append_mirror_pts(
            lat_ventricles, left_con_cand_pts, right_con_cand_pts
        )
        if len(left_con_cand_pts) > 0:
            left_con_cand_pts = left_con_cand_pts + left_pts_inv
        if len(right_con_cand_pts) > 0:
            right_con_cand_pts = right_con_cand_pts + right_pts_inv

        # Fit ellipses
        max_ecc = calculate_max_ecc(
            coronal_plane, lat_ventricles, left_con_cand_pts, right_con_cand_pts
        )

        results.append({"subject_id": subject_id, "max_ecc_lv": max_ecc})
        print(f"{subject_id}: MaxEccLV={max_ecc:.4f}")

    except Exception as e:
        print(f"{subject_id}: ERROR — {e}")
        results.append({"subject_id": subject_id, "max_ecc_lv": np.nan})

results_df = pd.DataFrame(results)
print(f"\nProcessed {len(results_df)} subjects")
print(f"Successful: {results_df['max_ecc_lv'].notna().sum()}")
print(f"Failed:     {results_df['max_ecc_lv'].isna().sum()}")

## Results: save CSV and inspect distribution

**Expected values:** Eccentricity ranges from 0 (circle) to <1 (line). Healthy lateral ventricles are moderately elongated (0.7–0.85). Enlarged ventricles in NPH or hydrocephalus tend to be rounder (lower eccentricity, ~0.5–0.7). A value of -100 indicates a failed ellipse fit for that side.

In [ ]:
# Merge with demographics if available
demo_df = None
if DEMOGRAPHICS_PATH and os.path.exists(DEMOGRAPHICS_PATH):
    ext = os.path.splitext(DEMOGRAPHICS_PATH)[1].lower()
    if ext in (".xlsx", ".xls"):
        demo_df = pd.read_excel(DEMOGRAPHICS_PATH)
    elif ext == ".csv":
        demo_df = pd.read_csv(DEMOGRAPHICS_PATH)
    elif ext in (".tsv", ".txt"):
        demo_df = pd.read_csv(DEMOGRAPHICS_PATH, sep="\t")

    if demo_df is not None:
        id_col = next(
            (c for c in demo_df.columns if c.upper() in ("ID", "SUBJECT_ID", "SUBJECTKEY", "SRC_SUBJECT_ID")),
            None,
        )
        if id_col:
            merge_cols = [id_col] + [
                c for c in demo_df.columns
                if c != id_col and c in ("Age", "M/F", "CDR", "Sex", "age", "sex", "interview_age")
            ]
            results_df = results_df.merge(
                demo_df[merge_cols], left_on="subject_id", right_on=id_col, how="left"
            )
        print(f"Merged with demographics: {list(results_df.columns)}")

# Summary statistics
valid_df = results_df[results_df["max_ecc_lv"] > 0]  # exclude -100 failed fits
print(f"\nMaxEccLV summary (n={len(valid_df)}, excluding failed fits):")
print(valid_df["max_ecc_lv"].describe().round(4))

In [ ]:
# Plot distribution and optional CDR stratification
has_cdr = "CDR" in results_df.columns and results_df["CDR"].notna().any()
n_plots = 1 + int(has_cdr)
fig, axes = plt.subplots(1, n_plots, figsize=(6 * n_plots, 5))
if n_plots == 1:
    axes = [axes]

# Distribution histogram
axes[0].hist(valid_df["max_ecc_lv"].dropna(), bins=20, edgecolor="black")
axes[0].set_xlabel("MaxEccLV")
axes[0].set_ylabel("Count")
axes[0].set_title("MaxEccLV Distribution")
axes[0].axvline(valid_df["max_ecc_lv"].median(), color="red", linestyle="--", label="median")
axes[0].legend()

# CDR stratification (if available)
if has_cdr:
    for cdr_val in sorted(results_df["CDR"].dropna().unique()):
        subset = results_df[(results_df["CDR"] == cdr_val) & (results_df["max_ecc_lv"] > 0)]["max_ecc_lv"]
        if len(subset) > 0:
            axes[1].hist(subset, bins=15, alpha=0.5, label=f"CDR {cdr_val}")
    axes[1].set_xlabel("MaxEccLV")
    axes[1].set_ylabel("Count")
    axes[1].set_title("MaxEccLV by Dementia Severity (CDR)")
    axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
output_path = os.path.join(OUTPUT_DIR, f"max_ecc_lv_{DATASET}.csv")
results_df.to_csv(output_path, index=False)
print(f"Results saved to {output_path}")
results_df.head(10)